In [ ]:
import geopandas as gpd
import sys

In [ ]:
sys.path.append('../src/nlr_ghe_screen/processing/')
from ms_cross_validation import (
    extract_quadkeys,
    get_ms_building_data
)

sys.path.append('../src/nlr_ghe_screen/geojson/')
from parse_geojson import (
    parse_api_response
)

sys.path.append('../src/nlr_ghe_screen/api/')
from osm_api import (
    get_overpass_no_splitting
)

In [ ]:
TEST_BBOX = (47.48, -122.45, 47.73, -122.22)
response = get_overpass_no_splitting(TEST_BBOX)

In [ ]:
gdf = parse_api_response(response, TEST_BBOX)
gdf.head()

In [ ]:
quadkeys = extract_quadkeys(TEST_BBOX)
ms_buildings = get_ms_building_data(quadkeys, TEST_BBOX)

In [ ]:
gdf = gdf.to_crs(ms_buildings.crs)

# Join each point to the footprint it falls in
fp = ms_buildings[['geometry']]
joined = gpd.sjoin(gdf, fp, how='left', predicate='within')

# Swap the point geometry for the footprint geometry
matched = joined[joined['index_right'].notna()].copy()
matched['geometry'] = ms_buildings.geometry.loc[matched['index_right'].astype(int)].values
matched = matched.drop(columns='index_right').set_geometry('geometry')

In [ ]:
# Ensure that there is only one polygon for each OSM row
# Two points that had overlapped the same polygon would create duplicated indices
matched = matched[~matched.index.duplicated(keep='first')]

# Overwrite the geometry for matched rows only
new_geom = gdf.geometry.copy()
new_geom.loc[matched.index] = matched.geometry
result = gdf.set_geometry(new_geom)

In [ ]:
result.explore()